<a href="https://www.kaggle.com/code/srikanthmachiraju/raft-finetuning-slm?scriptVersionId=317109933" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# RAFT Fine-Tuning 

## Overview

**RAFT (Retrieval-Augmented Fine-Tuning)** is a supervised fine-tuning strategy that trains a language model to answer questions from a retrieved context that contains both the relevant *oracle* passage and several *distractor* documents. This teaches the model to identify and reason from the correct source rather than relying on parametric memory — directly improving RAG pipeline performance at inference time without any architecture changes.

**Reference:** *RAFT: Adapting Language Model to Domain Specific RAG* (Zhang et al., 2024).

---

### End-to-End Pipeline

```
PDF documents
    └─► pdf_to_chunks.py     (GPT-4o vision → per-page .txt files)
    └─► raft_datagen.py      (GPT-4o → Q/A/D triplets → JSONL)
    └─► this notebook        (Unsloth + LoRA → fine-tuned Llama-3.2-1B)
```

### Dataset Schema

Each JSONL record contains:

| Field | Description |
|-------|-------------|
| `question` | Synthetic question generated from a document chunk |
| `context` | Dict with `sentences` — oracle + `num_distract` distractor passages (shuffled) |
| `oracle_context` | The single chunk that actually answers the question |
| `instruction` | Full `<DOCUMENT>…</DOCUMENT>` prompt fed to the model |
| `cot_answer` | Chain-of-thought answer with `##begin_quote##` citations and final `<ANSWER>:` tag |

### Key Training Hyperparameters

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| `max_seq_length` | 2048 | Covers instruction + 4 docs + CoT answer |
| `load_in_4bit` | True | NF4 quantisation — ~4× memory reduction |
| LoRA rank `r` | 16 | Balances expressivity vs. parameter count |
| `lora_alpha` | 16 | Effective scale = alpha/r = 1.0 |
| `learning_rate` | 2e-5 | Conservative for an instruction-tuned base |
| `gradient_accumulation_steps` | 8 | Effective batch size = 2 × 8 = 16 |
| `lr_scheduler_type` | cosine | Smooth decay; standard for short runs |

---

## Setup: Install Dependencies

In [1]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
try: 
    import subprocess; 
    is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: 
    is_t4 = False
_vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton} "huggingface_hub>=0.34.0" "datasets==4.3.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2
!uv pip install loguru sqlglot sqlparse swifter

## Step 1: Load the RAFT Dataset

Load the pre-generated RAFT training and evaluation splits from disk. Each split is a JSONL file where every record is a Q/A/D triplet.

- **`train.jsonl`** — 80% of generated examples, used for gradient updates  
- **`test.jsonl`** — held-out 20%, used for evaluation loss during training  

`tiktoken` is imported for optional token-count analysis of the dataset before training.

In [2]:
import os
import tiktoken
import json
import pandas as pd

MAX_SEQ_LEN = 2048 

training_data_dir = '/kaggle/input/datasets/srikanthmachiraju/raft-finetuning-dataset/'
train_df = pd.read_json(os.path.join(training_data_dir, "train.jsonl"), lines=True)
test_df = pd.read_json(os.path.join(training_data_dir, "test.jsonl"), lines=True)
print(f"Number of samples: train={len(train_df)}, test={len(test_df)}")
train_df.head()

Number of samples: train=507, test=64


,id,type,question,context,oracle_context,cot_answer,instruction
0,seed_task_83,distractor,What ServiceNow document outlines supported sk...,"{'title': [['placeholder_title', 'placeholder_...",## Evaluate and configure AI model options on ...,### Step-by-Step Reasoning:\n\n1. **Understand...,"<DOCUMENT>Additionally, it involves disabling ..."
1,seed_task_60,oracle,What does the ServiceNow Access Control plugin...,"{'title': [['placeholder_title', 'placeholder_...",## Activate the ServiceNow Access Control (SNA...,Step-by-step reasoning:\n\n1. To answer the qu...,<DOCUMENT>### Manage the organization’s mobile...
2,seed_task_41,oracle,What should password policies address in both ...,"{'title': [['placeholder_title', 'placeholder_...",#### Configure and enforce strong password pol...,Step-by-step reasoning:\n\n1. To answer the qu...,<DOCUMENT>#### Configure and enforce strong pa...
3,seed_task_72,oracle,Does ServiceNow provide free encryption option...,"{'title': [['placeholder_title', 'placeholder_...",```markdown\n# Data at rest\n\n## Customer res...,### Step-by-Step Reasoning:\n\n1. The question...,<DOCUMENT>```markdown\n# Email security\n\n## ...
4,seed_task_126,oracle,Where can ServiceNow security contacts be upda...,"{'title': [['placeholder_title', 'placeholder_...",#### Ensure security communications from Servi...,### Step-by-Step Reasoning:\n\n1. The question...,<DOCUMENT>ServiceNow publishes Security Adviso...


In [3]:
train_df[train_df["type"]=="distractor"].head()

,id,type,question,context,oracle_context,cot_answer,instruction
0,seed_task_83,distractor,What ServiceNow document outlines supported sk...,"{'title': [['placeholder_title', 'placeholder_...",## Evaluate and configure AI model options on ...,### Step-by-Step Reasoning:\n\n1. **Understand...,"<DOCUMENT>Additionally, it involves disabling ..."
5,seed_task_94,distractor,What authentication method is recommended for ...,"{'title': [['placeholder_title', 'placeholder_...",### Enforce network security for MID Server\nL...,### Step-by-Step Reasoning:\n\n1. **Understand...,<DOCUMENT>Establish a process involving releva...
11,seed_task_1,distractor,How are best practices organized within each s...,"{'title': [['placeholder_title', 'placeholder_...",```markdown\n# Introduction\n\n### Video: Secu...,Step-by-step reasoning:\n\n1. **Understand the...,<DOCUMENT>Administrators should collaborate cl...
15,seed_task_79,distractor,How can AI risks be assessed according to the ...,"{'title': [['placeholder_title', 'placeholder_...",### Additionally:\n- **Document operational bo...,Step-by-step reasoning:\n\n1. To assess AI ris...,<DOCUMENT>Customers should ensure that the [pr...
23,seed_task_16,distractor,What is a recommended practice for defining da...,"{'title': [['placeholder_title', 'placeholder_...",---\n\n### ServiceNow responsibility\nAs the d...,### Step-by-Step Reasoning:\n\n1. **Understand...,<DOCUMENT>The assessment typically spans sever...


## Step 2: Convert to HuggingFace `Dataset` Format

Convert the pandas DataFrames into HuggingFace `Dataset` objects. This enables:
- Efficient batched `.map()` transformations in subsequent steps
- Compatibility with `SFTTrainer` which expects HF datasets
- Arrow-backed memory mapping for large datasets

In [4]:
from datasets import Dataset

# Convert your pandas DataFrames to HF Datasets
train_dataset = Dataset.from_pandas(train_df)
eval_dataset  = Dataset.from_pandas(test_df)

# Check the structure
print(train_dataset)
print(train_dataset.column_names)
# print(train_dataset[0])   # view one example

Dataset({
    features: ['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction'],
    num_rows: 507
})
['id', 'type', 'question', 'context', 'oracle_context', 'cot_answer', 'instruction']


## Step 3: Load Base Model and Attach LoRA Adapters

Load **Llama-3.2-1B-Instruct** using [Unsloth](https://github.com/unslothai/unsloth)'s optimised `FastLanguageModel`, then attach **LoRA (Low-Rank Adaptation)** adapters to the attention and MLP projection layers.

### Why LoRA?
Full fine-tuning of a 1B parameter model requires updating all weights (~4 GB in fp16). LoRA instead injects small trainable rank decomposition matrices (`r=16`) into each target layer. This reduces trainable parameters by **~99%** while preserving most of the base model's capabilities.

### Target modules
All six projection matrices are adapted — `q/k/v/o_proj` (attention) and `gate/up/down_proj` (MLP) — which gives the model maximum flexibility to learn the RAFT task format.

### Memory optimisation
- **4-bit NF4 quantisation** reduces the frozen base weights to ~500 MB GPU memory  
- **`use_gradient_checkpointing="unsloth"`** trades recomputation for ~30% less VRAM during the backward pass

In [5]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

lora_rank = 8 # reduced to 8 after noticing overfitting

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, 
    dtype=None,
)

# Recommended for Llama-3.2
tokenizer = get_chat_template(
    tokenizer, 
    chat_template="llama-3.2"   # or "llama-3.2" if available in your Unsloth version
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 2 * lora_rank,
    lora_dropout = 0, # set to 0 for optimized training
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 2025,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-05-06 16:22:43.207333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778084563.442605      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778084563.508674      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778084564.061959      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778084564.061999      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778084564.062002      23 computation_placer.cc:177] computation placer alr

INFO 05-06 16:23:17 [__init__.py:244] Automatically detected platform cuda.
ERROR 05-06 16:23:21 [fa_utils.py:57] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth 2026.5.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


## Step 4: Apply Chat Template Formatting

Transform each raw Q/A/D record into a single text string using the **Llama-3.2 chat template**. This step is critical because the model was instruction-tuned with a specific prompt structure; maintaining that structure during fine-tuning prevents catastrophic forgetting of the base chat format.

### Prompt structure per example

```
<|system|>  You are a helpful assistant…
<|user|>              
            ### Context: <DOCUMENT>…</DOCUMENT> prompt
<|assistant|> <chain-of-thought reasoning> with ##begin_quote## and ##end_quote##
             <ANSWER>: …
```

### Why include distractors in the prompt?
RAFT deliberately exposes the model to noisy context during training. With probability `p=0.8` the oracle is present; with `p=0.2` it is replaced. This teaches robustness — the model learns to either find the answer or correctly abstain.

### Column cleanup
`remove_columns=train_dataset.column_names` drops all original columns, leaving only `text`. This avoids padding mismatches in the collator when column schemas differ between examples.

In [6]:
_SYSTEM_PROMPT = "You are a helpful assistant that answers questions using the provided context ONLY. \
Given the question, context and answer above, provide a logical reasoning for that answer. Please use the format of: \
### Step-by-step reasoning: \
##begin_quote## [Relevant text 1] ##end_quote## \
##begin_quote## [Relevant text 2] ##end_quote## \
<ANSWER>Answer here...</ANSWER>" 

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

def formatting_prompts_func(examples):
    texts = []
    for qn, ctx, oracle, instr, ans, _type in zip(
        examples["question"],
        examples["context"],
        examples["oracle_context"],
        examples["instruction"],
        examples["cot_answer"],
        examples["type"]
    ):
        
        messages = [
            {"role": "system", "content": _SYSTEM_PROMPT },
            {"role": "user", "content": f"<Retrieved Documents>: \n{instr}"}, # Context + Question
            {"role": "assistant", "content": f"{ans}"}   # COT style answer
        ]
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        texts.append(text) # + tokenizer.eos_token)
    
    return {"text": texts}

# Apply formatting
train_ds = train_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_dataset.column_names   # Clean old columns, keep only "text"
)

eval_ds = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=eval_dataset.column_names
)

# Verify
train_ds[0]

Map:   0%|          | 0/507 [00:00<?, ? examples/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are a helpful assistant that answers questions using the provided context ONLY. Given the question, context and answer above, provide a logical reasoning for that answer. Please use the format of: ### Step-by-step reasoning: ##begin_quote## [Relevant text 1] ##end_quote## ##begin_quote## [Relevant text 2] ##end_quote## <ANSWER>Answer here...</ANSWER><|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<Retrieved Documents>: \n<DOCUMENT>Additionally, it involves disabling local logins to enhance security. By default, MFA is enabled for all local logins, and customers must manage exceptions and monitor to ensure this default setting is maintained. The authentication mechanism plays a crucial role by verifying the identity of users accessing a ServiceNow instance and subsequently authorizing them to access features that correspond to their role o

## Step 5: Cache Formatted Datasets to Disk

Persist the formatted datasets in HuggingFace Arrow format. This is useful when:
- Iterating on training hyperparameters without re-running the expensive formatting step
- Re-using the exact same formatted data across kernel restarts

The next cell shows how to reload from disk, skipping Steps 1–4 entirely.

In [7]:
# After creating train_ds and eval_ds, save them
train_ds.save_to_disk("raft_train_hf")
eval_ds.save_to_disk("raft_eval_hf")

Saving the dataset (0/1 shards):   0%|          | 0/507 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/64 [00:00<?, ? examples/s]

In [8]:
# Later, load them directly (no need to convert again)
from datasets import load_from_disk
train_ds = load_from_disk("raft_train_hf")
eval_ds = load_from_disk("raft_eval_hf")

## Step 6: Configure and Initialise the SFT Trainer

Configure `trl.SFTTrainer` with `transformers.TrainingArguments`. Key decisions:

| Argument | Value | Notes |
|----------|-------|-------|
| `per_device_train_batch_size` | 2 | Constrained by GPU VRAM with 4-bit model |
| `gradient_accumulation_steps` | 8 | Effective batch = 16; smooths gradient noise |
| `num_train_epochs` | 1 | Single pass — RAFT data quality > quantity |
| `learning_rate` | 2e-5 | Below the typical 5e-5 to protect instruction-following |
| `fp16` | True | Mixed-precision training; compatible with T4/A10 |
| `eval_strategy` | steps (every 5) | Frequent eval to detect divergence early |
| `optim` | adamw_torch | Decoupled weight decay; preferred over legacy adamw |
| `lr_scheduler_type` | cosine | Smooth warmup-free decay for short runs |
| `save_strategy` | no | Model is saved explicitly in Step 10 after merging |

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth.chat_templates import train_on_responses_only

training_args = TrainingArguments(
    output_dir="llama32_1bn_instruct_raft", #This will also be used as your huggingfacehub model id name
    report_to="none", #Leave this to be blank if you don't want to use wandb
    per_device_train_batch_size=2,    # small batches if quantized
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,
    warmup_steps = 10,
    max_steps = 120,
    learning_rate=2e-5,
    save_strategy="no",
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    eval_strategy="steps",
    eval_steps=10,
    logging_steps=1,
    seed=42,
    optim="adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds, 
    args=training_args,
    dataset_text_field="text",
    packing = False, # Can make training 5x faster for short sequences.
)   

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/507 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/64 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/507 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/507 [00:00<?, ? examples/s]

Unsloth: Removed 7 out of 507 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=8):   0%|          | 0/64 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/64 [00:00<?, ? examples/s]

In [10]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are a helpful assistant that answers questions using the provided context ONLY. Given the question, context and answer above, provide a logical reasoning for that answer. Please use the format of: ### Step-by-step reasoning: ##begin_quote## [Relevant text 1] ##end_quote## ##begin_quote## [Relevant text 2] ##end_quote## <ANSWER>Answer here...</ANSWER><|eot_id|><|start_header_id|>user<|end_header_id|>\n\n<Retrieved Documents>: \n<DOCUMENT>Additionally, it involves disabling local logins to enhance security. By default, MFA is enabled for all local logins, and customers must manage exceptions and monitor to ensure this default setting is maintained. The authentication mechanism plays a crucial role by verifying the identity of users accessing a ServiceNow instance and subsequently authorizing them to access features that correspond to thei

In [11]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    ### Step-by-Step Reasoning:\n\n1. **Understand the Question**: The question asks for the name of the ServiceNow document that outlines supported skills and agents for Large Language Models (LLMs)

## Step 7: GPU Memory Snapshot (Pre-Training)

Record baseline GPU memory stats before training begins. These values are used post-training to calculate peak memory consumption and validate that the configuration fits within the available VRAM budget. If `start_gpu_memory / max_memory > 0.8`, consider reducing `per_device_train_batch_size` or enabling `load_in_8bit`.

In [12]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.219 GB of memory reserved.


## Step 8: Train the Model

Launch the training loop. `trainer.train()` returns a `TrainOutput` object containing:
- `global_step` — total number of optimiser steps taken  
- `training_loss` — average loss over the run  
- `metrics` — timing and throughput statistics  

Training progress and eval loss are logged every 5 steps. Monitor for:
- **Decreasing train loss** — confirms the model is learning the RAFT format  
- **Eval loss tracking train loss** — large divergence indicates overfitting; consider reducing epochs or increasing dropout

In [13]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 15 | Total steps = 120
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
10,1.131300,1.062826
20,0.879900,0.845402
30,0.734500,0.725385
40,0.680600,0.660273
50,0.604200,0.621200
60,0.651000,0.597733
70,0.556600,0.582312
80,0.578800,0.572159
90,0.531500,0.564664
100,0.514700,0.560127


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## Step 9: Review Training Statistics

Inspect the `TrainOutput` object. Key metrics to review:

- **`train_loss`** — final training loss; should be below 1.0 for a well-fitted RAFT model  
- **`train_runtime`** — total wall-clock training time in seconds  
- **`train_samples_per_second`** — throughput; use this to extrapolate cost for longer runs  
- **`train_steps_per_second`** — compare against expected steps given your batch config

In [14]:
trainer_stats

TrainOutput(global_step=120, training_loss=0.701356726884842, metrics={'train_runtime': 4119.4369, 'train_samples_per_second': 1.864, 'train_steps_per_second': 0.029, 'total_flos': 4.494644635661107e+16, 'train_loss': 0.701356726884842, 'epoch': 15.0})

Verify GPU usage after training

In [15]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
9.836 GB of memory reserved.


In [16]:
import gc; gc.collect()

127

## Step 10: Save Merged Model in 16-bit

Merge the LoRA adapter weights back into the base model and save as a single 16-bit checkpoint. This produces a standard HuggingFace model directory that can be:
- Loaded with `AutoModelForCausalLM.from_pretrained()` — no LoRA dependency required
- Quantised further (GGUF, GPTQ) for local deployment
- Pushed directly to the Hub

`save_method="merged_16bit"` is preferred over `lora_only` for portability and over `merged_4bit` when upload size is not a constraint.

In [17]:
model.save_pretrained_merged(
    save_directory = "llama32_1bn_instruct_raft",     
    tokenizer = tokenizer,
    save_method = "merged_16bit",        
)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:07<00:00,  7.48s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:16<00:00, 16.37s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/llama32_1bn_instruct_raft`


## Step 11: Authenticate with Hugging Face Hub

Retrieve the HF API token from Kaggle Secrets and authenticate. This scopes authentication to the current session without persisting the token to disk. The token requires **write** access to the target repository.

In [18]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

## Step 12: Push Model and Tokenizer to Hugging Face Hub

Upload the merged model weights and tokenizer to the specified Hub repository. Both must be pushed — the tokenizer carries the chat template configuration which is required for correct inference.

In [19]:
hf_model_path = "sriksmachi/llama32_1bn_instruct_raft"

# Use model.push_to_hub() to upload
model.push_to_hub(hf_model_path)

# Don't forget to push the tokenizer as well
tokenizer.push_to_hub(hf_model_path)

README.md:   0%|          | 0.00/584 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/sriksmachi/llama32_1bn_instruct_raft


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


## Step 13: Pull Model from HF and generate answers

Pulls the model and tokenizer from HF, and generate answers. It is important to enable the inference mode

In [20]:
from unsloth import FastLanguageModel

fn_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=hf_model_path,
    max_seq_length=MAX_SEQ_LEN,
)

FastLanguageModel.for_inference(fn_model)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


adapter_model.safetensors:   0%|          | 0.00/22.6M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

In [21]:
from transformers import TextStreamer
from tqdm import tqdm

validation_df = pd.read_json(os.path.join(training_data_dir, "validation.jsonl"), lines=True)

baseline_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = MAX_SEQ_LEN, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
)

FastLanguageModel.for_inference(baseline_model)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2. vLLM: 0.9.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
 

## Step 14: Model Evaluation for oracle and distractor samples

In [22]:
MAX_NEW_TOKENS = 512

def generate_answer(instruction, model):
    """Generate an answer using a HuggingFace text-generation pipeline."""
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": f"<Retrieved Documents>: {instruction}"}, # Context + Question
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens = MAX_NEW_TOKENS, use_cache = False, temperature = 0.1, min_p = 0.1,do_sample = False)
    input_length = inputs.shape[1]
    generated_tokens = outputs[0][input_length:]
    generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=False)
    return generated_text

# Response for oracle docs
oracle_sample = validation_df[validation_df["type"] == "oracle"].iloc[1]
print(f"====================Question===================\n")
print(f"{oracle_sample["question"]}")
print(f"===================INSTRUCTION================\n")
print(f"{oracle_sample["instruction"]}")
print(f"=====================Baseline Model Response===================\n")
print(generate_answer(oracle_sample["instruction"], baseline_model))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


====================Question===================

Why should customers regularly review the ServiceNow security advisories page?
===================INSTRUCTION================

<DOCUMENT>ServiceNow publishes Security Advisories to inform customers of relevant vulnerabilities, mitigations, and risks affecting the ServiceNow AI Platform and supporting cloud environments. Through the Patching Program, ServiceNow delivers regular updates that include security, performance, and functional fixes to customer instances. ServiceNow enforces an End-of-Life (EOL) policy to ensure that only supported versions receive updates, helping maintain a secure and stable platform for all customers.

### Best practices  

#### Regularly review the ServiceNow security advisories page  
These advisories often highlight vulnerabilities addressed through patches or upgrades, ensuring customers can apply updates promptly to maintain security and compliance. Work with stakeholders, such as the security team, when 

In [23]:
%%time
print(f"=====================Finetuned Model Response===================\n")
print(generate_answer(oracle_sample["instruction"], fn_model))

=====================Finetuned Model Response===================

### Step-by-Step Reasoning:

1. **Understand the question**: The question asks why customers should regularly review the ServiceNow security advisories page. This means we need to identify the purpose or benefit of reviewing these advisories.

2. **Locate relevant information in the context**: The context provides details about the purpose of reviewing ServiceNow security advisories. Specifically, it mentions that these advisories often highlight vulnerabilities addressed through patches or upgrades, ensuring customers can apply updates promptly.

3. **Extract relevant sentences**: From the context, the following sentence is relevant:
   ##begin_quote## These advisories often highlight vulnerabilities addressed through patches or upgrades, ensuring customers can apply updates promptly to maintain security and compliance. ##end_quote##

4. **Interpret the extracted information**: The sentence explains that the purpose of 

### Distractor Sample

In [24]:
# Response for oracle docs
for i in range(2):
    distractor_sample = validation_df[validation_df["type"] == "distractor"].iloc[i]
    print(f"===========================SAMPLE {i}======================\n")
    print(f"===================INSTRUCTION================\n")
    print(f"{distractor_sample["instruction"]}")
    print(f"=====================Baseline Model Response===================\n")
    print(generate_answer(distractor_sample["instruction"], baseline_model))
    print(f"=====================Finetuned Model Response===================\n")
    print(generate_answer(distractor_sample["instruction"], fn_model))
    print(f"===========================================================\n\n")

===========================SAMPLE 0======================

===================INSTRUCTION================

<DOCUMENT>### Consider your AI security strategy:
- **Engage key stakeholders**: Identify all stakeholders involved throughout the AI lifecycle and clearly define and document their roles and responsibilities.
- **Define system purpose**: Clearly articulate the task and intended purpose of each AI system.
- **Enable human oversight**: Establish procedures that allow authorized personnel to interrupt, override, or halt system operations when necessary.
- **Provide contestability**: Implement mechanisms for challenging automated decisions, ensuring human operators can review actions taken by AI systems and, if appropriate, supplement them.

Customers should establish accountability for AI-driven processes and implement appropriate controls to guard against unintended outcomes.</DOCUMENT>
<DOCUMENT>```markdown
# Secure coding

## Customer responsibility
Customers are responsible for 